# ch01 — TSAD 문제 정의

이상 유형(point/contextual/collective)을 직접 만들어 본다.
이론: [docs/learn/ch01](../docs/learn/ch01_problem.md)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
# 이상 유형별 합성 데이터
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, kind in zip(axes, ["spike", "level_shift", "contextual"]):
    ds = generate_synthetic(n_test=800, n_events=3, anomaly_kinds=[kind], seed=1)
    ax.plot(ds.test[:, 0], lw=0.8)
    ax.fill_between(np.arange(800), *ax.get_ylim(), where=ds.labels.astype(bool),
                    alpha=0.25, color="red")
    ax.set_title(f"anomaly kind = {kind}  (rate={ds.anomaly_rate:.3f})")
plt.tight_layout()

In [ ]:
# contamination: train 오염이 zscore baseline에 미치는 영향
from tsad_forge.evaluation.metrics import compute_metrics
from tsad_forge.models.registry import get_model

for cont in [0.0, 1.0, 3.0]:
    ds = generate_synthetic(seed=3, contamination=cont)
    scores = get_model("zscore").fit(ds.train).score(ds.test)
    m = compute_metrics(scores, ds.labels)
    print(f"contamination={cont}: VUS-PR={m['vus_pr']:.3f}  AUC-PR={m['auc_pr']:.3f}")